# Histopathology Cell Classification: Showcase Notebook

Two related tasks on 27x27 RGB H&E tiles of colon cells, evaluated on **unseen patients**:

| Task | Type | Data | Target (macro-F1) |
|---|---|---|---|
| `isCancerous` | binary | 20,280 tiles, 98 patients | 0.90 |
| `cellType` | 4-class (fibroblast, inflammatory, epithelial, others) | 9,896 tiles, 60 patients | 0.60 |

All reusable logic lives in [`src/`](../src); this notebook is the narrative layer (experiments, plots, interpretation).

**Results at a glance** (final models, held-out test patients; from the original run, small differences are expected across hardware):

| Task | Final model | Test macro-F1 | Target met? |
|---|---|---|---|
| `isCancerous` | WiderCNN + augmentation (15 ep) | **0.854** | No (0.90) |
| `cellType` | SmallCNN, transferred from isCancerous + augmentation (12 ep) | **0.516** | No (0.60) |

**Contents**
1. Setup and data check
2. EDA (class balance, label structure, sample tiles, PCA + k-means)
3. Patient-disjoint splits
4. Classical baselines
5. CNN baseline and learning-rate sweep
6. Advanced technique 1: data augmentation
7. Advanced technique 2: intra-dataset transfer learning (+ class-weighted loss)
8. Pushing the isCancerous ceiling (longer training, wider network)
9. Model comparison and selection
10. Held-out test evaluation
11. Limitations and next steps

> **Runtime:** about 25-35 min on CPU. Set `QUICK = True` below for a ~2 min smoke run on 10% of the patients (numbers will not be meaningful).


## 1. Setup and data check

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

# Make `src/` importable whether the notebook is launched from the repo root or notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config as C
from src.data import (build_task_frame, check_patient_overlap, flatten_images, load_images_chw,
                      load_images_flat, load_label_tables)
from src.evaluate import (evaluate_cnn, evaluate_sklearn, per_class_table, plot_confusion_matrix,
                          plot_generalisation_gap, plot_histories, plot_lr_sweep, predict_cnn,
                          results_table, save_predictions, build_test_report)
from src.models import (SmallCNN, WiderCNN, build_cnn, build_sklearn_model, count_parameters,
                        make_target_from_source)
from src.splits import (cross_task_overlap, describe_split, make_patient_disjoint_split,
                        subsample_patients)
from src.train import compute_class_weights, fit_cnn, fit_sklearn, learning_rate_sweep
from src.utils import get_device, seed_everything

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid", palette="Set2")

QUICK = False  # True -> 10% of patients, 1 epoch per model (smoke test only)
E = lambda n: 1 if QUICK else n  # epoch budget helper

seed_everything(C.SEED)
DEVICE = get_device()
print("Device:", DEVICE)

In [ ]:
main_df, extra_df = load_label_tables(C.DATA_DIR)
print(f"main_df : {main_df.shape}  columns={list(main_df.columns)}")
print(f"extra_df: {extra_df.shape}  columns={list(extra_df.columns)}")

n_overlap = check_patient_overlap(main_df, extra_df)
print(f"\nPatients: main={main_df.patientID.nunique()}, extra={extra_df.patientID.nunique()}, overlap={n_overlap} (must be 0)")
assert n_overlap == 0

frames = {t: build_task_frame(t, main_df, extra_df) for t in C.TASKS}
if QUICK:
    frames = {t: subsample_patients(f, 0.10) for t, f in frames.items()}
print({t: f"{len(f):,} images / {f.patientID.nunique()} patients" for t, f in frames.items()})

## 2. Exploratory data analysis

**Questions:** how imbalanced are the targets, how do the two labels relate, and do raw pixels carry usable signal without learning?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

can = frames["isCancerous"]["isCancerous"].value_counts().sort_index()
axes[0].bar(C.CANCER_NAMES, can.values, color=["steelblue", "coral"], edgecolor="black")
axes[0].set_title(f"isCancerous (N={can.sum():,})", fontweight="bold")
for i, v in enumerate(can.values):
    axes[0].text(i, v, f"{v:,} ({v / can.sum():.1%})", ha="center", va="bottom", fontsize=9)

ct = frames["cellType"]["cellTypeName"].value_counts()
axes[1].bar(ct.index, ct.values, color=sns.color_palette("Set2", len(ct)), edgecolor="black")
axes[1].set_title(f"cellTypeName (N={ct.sum():,})", fontweight="bold")
axes[1].tick_params(axis="x", rotation=15)
for i, v in enumerate(ct.values):
    axes[1].text(i, v, f"{v:,} ({v / ct.sum():.1%})", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

Both targets are moderately imbalanced (about 65/35 for `isCancerous`; 41/26/19/14 % for `cellType`), so **macro-averaged F1** is used as the primary metric: it weights every class equally and cannot be gamed by ignoring a minority class.

In [ ]:
display(pd.crosstab(frames["cellType"]["cellTypeName"], frames["cellType"]["isCancerous"], margins=True))

In the main data the labels are **deterministically linked**: every epithelial cell is cancerous and nothing else is. So `isCancerous` on the main subset is effectively "is this epithelial?", and the hard part of `cellType` is separating the three non-cancerous classes. (This also matters for transfer learning, see the limitations section.)

In [ ]:
rng = np.random.default_rng(C.SEED)
ct_df = frames["cellType"]
fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for r, cls in enumerate(C.CTYPE_NAMES):
    names = ct_df.loc[ct_df.cellTypeName == cls, "ImageName"].values
    for c, name in enumerate(rng.choice(names, size=min(4, len(names)), replace=False)):
        axes[r, c].imshow(Image.open(C.IMG_DIR / name)); axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
    axes[r, 0].set_ylabel(cls, rotation=0, ha="right", va="center")
fig.suptitle("Random samples per cellTypeName", fontweight="bold"); plt.tight_layout(); plt.show()

### Unsupervised check: PCA + k-means on raw pixels
If raw pixel statistics separated the classes, k-means clusters would align with the labels (measured by Adjusted Rand Index).

In [ ]:
X_pix = load_images_flat(ct_df)
X_pca = PCA(n_components=50, random_state=C.SEED).fit_transform(StandardScaler().fit_transform(X_pix))
clusters = KMeans(n_clusters=4, random_state=C.SEED, n_init=10).fit_predict(X_pca)

print(f"ARI k-means vs cellType   : {adjusted_rand_score(ct_df.cellType, clusters):.4f}")
print(f"ARI k-means vs isCancerous: {adjusted_rand_score(ct_df.isCancerous, clusters):.4f}")

fig, ax = plt.subplots(figsize=(6.5, 5))
for name in C.CTYPE_NAMES:
    m = ct_df.cellTypeName.values == name
    ax.scatter(X_pca[m, 0], X_pca[m, 1], s=2, alpha=0.35, label=name)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend(markerscale=4)
ax.set_title("PCA of raw pixels, coloured by cellType", fontweight="bold"); plt.show()

ARI is about 0.04-0.05 for both labels, i.e. near-random. **Raw pixels do not cluster by class**, which motivates models that *learn* representations (CNNs) rather than relying on flattened pixels.

## 3. Patient-disjoint splits

A random row-level split would put tiles from the same patient in train *and* test and measure memorisation. We use nested `GroupShuffleSplit` on `patientID` for a **60/20/20 split by patient**, built separately per task (they draw from different patient pools). Disjointness is asserted in code.

In [ ]:
splits = {t: make_patient_disjoint_split(f, seed=C.SEED) for t, f in frames.items()}
for t, s in splits.items():
    print(f"\n{t}")
    display(describe_split(s, C.TASKS[t].label_col))

print("\nCross-task overlap (isCancerous train patients that are cellType val/test patients):")
print(cross_task_overlap(splits["isCancerous"], splits["cellType"]))

Patient-level splitting makes class mix shift between splits (for example, the cancer rate differs between train, val and test). That is a real property of generalising to new patients, not a bug.

The overlap printed above matters later: both tasks are split independently, so a patient can be in the `isCancerous` training set *and* in the `cellType` test set.

In [ ]:
data = {}
for t, s in splits.items():
    lab = C.TASKS[t].label_col
    data[t] = {n: (load_images_chw(df), df[lab].values.astype("int64")) for n, df in zip(("train", "val", "test"), s)}
    print(t, {n: tuple(X.shape) for n, (X, _) in data[t].items()})

def arrays(task):
    (Xtr, ytr), (Xva, yva), (Xte, yte) = (data[task][k] for k in ("train", "val", "test"))
    return Xtr, ytr, Xva, yva, Xte, yte

results = {"isCancerous": {}, "cellType": {}}   # {task: {model: {train_f1, val_f1, ...}}}
hist = {"isCancerous": {}, "cellType": {}}      # {task: {model: training history}}

## 4. Classical baselines (flattened pixels)

| Model | Why |
|---|---|
| Logistic regression on PCA(50) | low-variance linear baseline (isCancerous) |
| SVM-RBF on PCA(50) | non-linear margin-based baseline (cellType) |
| Random forest on raw pixels | high-capacity non-parametric model |

In [ ]:
def run_sklearn(task, builder, label):
    Xtr, ytr, Xva, yva, *_ = arrays(task)
    Xtr_f, Xva_f = flatten_images(Xtr), flatten_images(Xva)
    clf, secs = fit_sklearn(build_sklearn_model(builder), Xtr_f, ytr)
    print(f"[{secs:.1f}s]", end=" ")
    results[task][label] = evaluate_sklearn(clf, Xtr_f, ytr, Xva_f, yva, label=label)

run_sklearn("isCancerous", "logreg_pca", "LogReg + PCA(50)")
run_sklearn("isCancerous", "random_forest", "Random Forest (flat pixels)")
run_sklearn("cellType", "svm_pca", "SVM-RBF + PCA(50)")
run_sklearn("cellType", "random_forest", "Random Forest (flat pixels)")

The random forest reaches train F1 of about 1.0 with a much lower validation F1 (classic **overfitting**: unpruned trees memorise pixels). The linear model has a small gap but a low ceiling.

## 5. CNN baseline and learning-rate sweep

`SmallCNN`: two double-conv blocks (3 to 32 to 64 channels, 3x3 kernels), max-pooling, 128-unit head with dropout (about 361k parameters). Weight sharing exploits spatial structure that flattening destroys.

In [ ]:
Xtr, ytr, Xva, yva, *_ = arrays("isCancerous")
base_cfg = C.TrainConfig(epochs=E(10))
print("SmallCNN parameters:", f"{count_parameters(SmallCNN(2)):,}")

can_base, hist["isCancerous"]["Small CNN (lr=1e-3)"] = fit_cnn(
    lambda: SmallCNN(2, base_cfg.dropout), Xtr, ytr, Xva, yva, base_cfg)
results["isCancerous"]["Small CNN (lr=1e-3)"] = evaluate_cnn(can_base, Xtr, ytr, Xva, yva, label="SmallCNN")

### Controlled experiment: Adam learning rate
Same architecture, seed and weight decay; only `lr` changes (5 epochs each). Theory predicts a U-shape: too high is unstable, too low is too slow.

In [ ]:
sweep = learning_rate_sweep(lambda: SmallCNN(2), Xtr, ytr, Xva, yva, lr_grid=C.LR_GRID, epochs=E(5))
print("Best learning rate:", f"{sweep['best_lr']:.0e}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
plot_lr_sweep(sweep, ax=axes[0])
plot_histories({f"lr={lr:.0e}": h for lr, h in sweep["histories"].items()}, key="val_f1",
               title="Validation F1 across epochs", ax=axes[1])
plt.tight_layout(); plt.show()

`lr=1e-3` is best and `1e-4` is close; `1e-2` is unstable early and `1e-5` barely starts learning in 5 epochs. `1e-3` is used from here on.

## 6. Advanced technique 1: data augmentation

Horizontal flip + vertical flip + rotation of up to 15 degrees, applied to the **training set only**. Cells have no canonical orientation, so the label is invariant to these transforms; augmentation encodes that prior and should shrink the generalisation gap.

In [ ]:
can_aug, hist["isCancerous"]["CNN + augmentation"] = fit_cnn(
    lambda: SmallCNN(2), Xtr, ytr, Xva, yva, C.TrainConfig(epochs=E(10)), augment=True)
results["isCancerous"]["CNN + augmentation"] = evaluate_cnn(can_aug, Xtr, ytr, Xva, yva, label="SmallCNN + aug")

Xt_tr, yt_tr, Xt_va, yt_va, *_ = arrays("cellType")
ct_base, hist["cellType"]["Small CNN (lr=1e-3)"] = fit_cnn(
    lambda: SmallCNN(4), Xt_tr, yt_tr, Xt_va, yt_va, C.TrainConfig(epochs=E(10)))
results["cellType"]["Small CNN (lr=1e-3)"] = evaluate_cnn(ct_base, Xt_tr, yt_tr, Xt_va, yt_va, label="cellType SmallCNN")

ct_aug, hist["cellType"]["CNN + augmentation"] = fit_cnn(
    lambda: SmallCNN(4), Xt_tr, yt_tr, Xt_va, yt_va, C.TrainConfig(epochs=E(10)), augment=True)
results["cellType"]["CNN + augmentation"] = evaluate_cnn(ct_aug, Xt_tr, yt_tr, Xt_va, yt_va, label="cellType SmallCNN + aug")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for row, task, b, a in [(0, "isCancerous", "Small CNN (lr=1e-3)", "CNN + augmentation"),
                        (1, "cellType", "Small CNN (lr=1e-3)", "CNN + augmentation")]:
    for col, key in enumerate(("f1", "loss")):
        ax = axes[row, col]
        for name, h, c in [("baseline", hist[task][b], "b"), ("augmented", hist[task][a], "r")]:
            ax.plot(h["epoch"], h[f"train_{key}"], f"{c}-o", ms=4, label=f"{name} train")
            ax.plot(h["epoch"], h[f"val_{key}"], f"{c}--s", ms=4, label=f"{name} val")
        ax.set_title(f"{task}: {key}", fontweight="bold"); ax.set_xlabel("Epoch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

Augmented *training* F1 sits below baseline (each batch is a new view), while validation F1 matches or exceeds baseline and the train/val gap narrows: the regularisation signature.

## 7. Advanced technique 2: intra-dataset transfer learning

The convolutional trunk of the `isCancerous` CNN (12k training tiles, same domain) initialises the `cellType` CNN (5.7k training tiles); only the head is re-initialised. Four regimes:

1. **Fine-tune all layers**
2. **Frozen trunk** (linear probe)
3. **Fine-tune + augmentation**
4. **Fine-tune + augmentation + class-weighted loss** (weights `N / (K * n_c)`)

> CIFAR-10 is deliberately *not* used as a source: it is natural images at 32x32, a poor domain match for H&E tiles.

In [ ]:
Xs_tr, ys_tr, Xs_va, ys_va, *_ = arrays("isCancerous")   # transfer source = SmallCNN trained on isCancerous (can_base)
transfer = lambda: make_target_from_source(can_base, num_classes=4)
cfg12 = C.TrainConfig(epochs=E(12))

variants = {
    "CNN + transfer (fine-tune)": dict(),
    "CNN + transfer (frozen)": dict(freeze_features=True),
    "CNN + transfer + augmentation": dict(augment=True),
    "CNN + transfer + aug + class-weighted": dict(
        augment=True, class_weights=compute_class_weights(yt_tr, 4, DEVICE)),
}
models_ct = {}
for name, kw in variants.items():
    print(f"\n--- {name} ---")
    models_ct[name], hist["cellType"][name] = fit_cnn(transfer, Xt_tr, yt_tr, Xt_va, yt_va, cfg12, **kw)
    results["cellType"][name] = evaluate_cnn(models_ct[name], Xt_tr, yt_tr, Xt_va, yt_va, label=name)

In [ ]:
ct_hist = {"baseline (scratch)": hist["cellType"]["Small CNN (lr=1e-3)"],
           "augmented only": hist["cellType"]["CNN + augmentation"],
           **{n: hist["cellType"][n] for n in variants}}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
plot_histories(ct_hist, "val_f1", target=C.TASKS["cellType"].target_macro_f1, title="cellType: validation F1", ax=axes[0])
plot_generalisation_gap(ct_hist, title="cellType: generalisation gap", ax=axes[1])
plt.tight_layout(); plt.show()

**Read the curves carefully.** Validation macro-F1 on `cellType` swings by 0.05-0.15 between consecutive epochs (only 12 validation patients), so a single "best epoch" number is optimistic. Compare *last-epoch* and *best-epoch* values below before drawing conclusions.

In [ ]:
rows = []
for name in variants:
    h = hist["cellType"][name]
    rows.append({"variant": name, "val F1 (last epoch)": h["val_f1"][-1], "val F1 (best epoch)": max(h["val_f1"]),
                 "best epoch": h["best_epoch"]})
display(pd.DataFrame(rows).set_index("variant").round(4))

ct_val_pred = lambda m: predict_cnn(m, Xt_va)
print("\nPer-class validation metrics (transfer + aug):")
display(per_class_table(yt_va, ct_val_pred(models_ct["CNN + transfer + augmentation"]), C.CTYPE_NAMES).round(3))
print("Per-class validation metrics (transfer + aug + class-weighted):")
display(per_class_table(yt_va, ct_val_pred(models_ct["CNN + transfer + aug + class-weighted"]), C.CTYPE_NAMES).round(3))

Class weighting lifts recall on `others` but costs precision there and on `epithelial`; macro-F1 does not improve. The `others` class is a heterogeneous catch-all, so the bottleneck is **visual signal, not class imbalance**.

## 8. Pushing the isCancerous ceiling

Two single-variable experiments on why validation F1 stalls below 0.90:
1. **Longer training:** SmallCNN + augmentation, 15 epochs.
2. **More capacity:** `WiderCNN` (about 4x parameters) + augmentation, 15 epochs.

In [ ]:
cfg15 = C.TrainConfig(epochs=E(15))
can_long, hist["isCancerous"]["CNN + aug (15 ep)"] = fit_cnn(lambda: SmallCNN(2), Xtr, ytr, Xva, yva, cfg15, augment=True)
results["isCancerous"]["CNN + aug (15 ep)"] = evaluate_cnn(can_long, Xtr, ytr, Xva, yva, label="SmallCNN + aug (15 ep)")

print(f"\nSmallCNN: {count_parameters(SmallCNN(2)):,} params | WiderCNN: {count_parameters(WiderCNN(2)):,} params")
can_wide, hist["isCancerous"]["WiderCNN + aug (15 ep)"] = fit_cnn(lambda: WiderCNN(2), Xtr, ytr, Xva, yva, cfg15, augment=True)
results["isCancerous"]["WiderCNN + aug (15 ep)"] = evaluate_cnn(can_wide, Xtr, ytr, Xva, yva, label="WiderCNN + aug (15 ep)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
runs = {k: hist["isCancerous"][k] for k in ("Small CNN (lr=1e-3)", "CNN + aug (15 ep)", "WiderCNN + aug (15 ep)")}
plot_histories(runs, "val_f1", target=C.TASKS["isCancerous"].target_macro_f1, title="isCancerous: validation F1", ax=axes[0])
plot_generalisation_gap(runs, title="isCancerous: generalisation gap", ax=axes[1])
plt.tight_layout(); plt.show()

Longer training helps slightly, but a 4x larger network adds almost nothing. Capacity is **not** the bottleneck: the 27x27 resolution and patient-level variability are.

## 9. Model comparison and selection

In [ ]:
for task, tbl in results_table(results).items():
    print(f"\n=== {task} (sorted by last-epoch validation macro-F1) ===")
    display(tbl.round(4))

**Do more complex models always win?** No. The random forest has the most capacity and the worst generalisation gap; `WiderCNN` has 4x the parameters of `SmallCNN` and ties it; a frozen trunk with one trainable layer beats several larger models early on. This is the bias-variance trade-off at work with limited patients.

**Final models** (as in the original analysis):
- `isCancerous`: `WiderCNN + aug (15 ep)`
- `cellType`: `CNN + transfer + augmentation`, chosen over the class-weighted variant by best-epoch validation F1 (0.694 vs 0.668). Note that selection by *best epoch* but evaluation of *last-epoch weights* is optimistic. The pipeline script exposes `--restore-best` to make the two consistent.

## 10. Held-out test evaluation

The test patients were never used for training, tuning or model selection. Weights tuned on validation are used directly (no retraining on train+val).

In [ ]:
final = {
    "isCancerous": ("WiderCNN + aug (15 ep)", can_wide),
    "cellType": ("CNN + transfer + augmentation", models_ct["CNN + transfer + augmentation"]),
}
reports = {}
for task, (name, model) in final.items():
    spec = C.TASKS[task]
    *_, Xte, yte = arrays(task)
    yp = predict_cnn(model, Xte)
    reports[task] = (yte, yp, build_test_report(yte, yp, spec.class_names, spec.target_macro_f1))
    r = reports[task][2]
    print(f"\n===== {task}: {name} =====")
    print(f"macro-F1 = {r['macro_f1']:.4f} (target {spec.target_macro_f1})  accuracy = {r['accuracy']:.4f}  target met: {r['target_met']}")
    print(r["classification_report"])
    if task == "isCancerous":
        print({k: round(r[k], 4) for k in ("sensitivity", "specificity", "ppv", "npv")})
    save_predictions(splits[task].test, yp, spec.label_col, C.OUT_DIR / f"test_predictions_{task}.csv", spec.class_names)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
yte, yp, r = reports["isCancerous"]
plot_confusion_matrix(yte, yp, C.CANCER_NAMES, f"isCancerous (test), macro-F1 = {r['macro_f1']:.3f}", "Blues", ax=axes[0])
yte, yp, r = reports["cellType"]
plot_confusion_matrix(yte, yp, C.CTYPE_NAMES, f"cellType (test), macro-F1 = {r['macro_f1']:.3f}", "Greens", ax=axes[1])
plt.tight_layout(); plt.show()

**isCancerous:** test macro-F1 about 0.85 (target 0.90 not met). Sensitivity about 0.89, specificity about 0.87. The val-to-test drop is small (about 0.02), and every CNN variant plateaus near 0.87 on validation.

**cellType:** test macro-F1 about 0.52 (target 0.60 not met). Epithelial is reliable (F1 about 0.83); `others` is the bottleneck (F1 about 0.12). Remember the last-epoch validation F1 of this model was about 0.59, not the 0.69 best-epoch peak, so the val-to-test gap is closer to 0.07 than 0.18.

## 11. Limitations and next steps

**Limitations**
- **Resolution ceiling:** 27x27 tiles discard the texture and context a pathologist uses.
- **Small patient pool for cellType:** 12 test patients; one or two atypical patients move the macro-F1 noticeably. Epoch-to-epoch validation F1 is very noisy.
- **Heterogeneous `others` class:** a catch-all without a coherent visual signature.
- **Possible transfer-source leakage:** the `isCancerous` source model was trained on patients who may also be `cellType` validation/test patients (check `cross_task_overlap` above; the pipeline's `--leakage-safe` flag removes them). Because cancer status equals "epithelial" in this data, such leakage could flatter the epithelial class.
- **Selection vs. evaluation mismatch:** best-epoch validation score used for selection, last-epoch weights evaluated.
- **Single institution, single split:** no external validation, no repeated or cross-validated patient splits, no confidence intervals.

**Ethics:** a false negative (missed cancer) costs far more than a false positive, so a deployed operating point should favour sensitivity. This is research code, not a diagnostic tool; it would at most support pathologist review.

**Next steps**
1. Re-run with `--leakage-safe --restore-best` and report both.
2. Grouped k-fold over patients with confidence intervals.
3. Higher-resolution inputs or stain-normalisation / colour augmentation.
4. Split `others` into homogeneous sub-classes; add threshold tuning for sensitivity.
